In [2]:
import random
import math
import sympy as sp

e_fix = 2**16+1 #=65537

def choose_e(phi):
    e = e_fix
    if math.gcd(e, phi) == 1:
        return e
    # fallback: pick random odd number
    while True:
        e = random.randrange(3, phi, 2)
        if math.gcd(e, phi) == 1:
            return e

# RSA demonstration

### Preparation of a public key

In [21]:
order = 2**4 #2**5 #2**10 #
p,q = tuple(sp.randprime(2**(order-1), 2**order) for _ in range(2))
print(p, q)

n = p*q
phi = (p-1)*(q-1)
e = choose_e(phi)

public_key = (n,e)
print(f"{public_key=}")

33637 35831
public_key=(1205247347, 65537)


In [22]:
phi, phi%(q-1)

(1205177880, 0)

### Encryption

In [23]:
#pw = (1234, 5678, 9012, 3456)
pw = tuple(random.randint(10**3, 10**4) for _ in range(4))
display(pw)

(2120, 6714, 2126, 3784)

In [24]:
encrypted_pw = tuple(num**public_key[1] % public_key[0] for num in pw)
print(f"{encrypted_pw=}")

encrypted_pw=(151605065, 1104562121, 1056128842, 321307975)


### Creation of the private key

In [25]:
### Euclid's algorithm

keys = ['quo', 'rem']
divmod_list = [dict(zip(keys, (0, val))) for val in [phi, e]]

def Euclids_algorithm_one_step(rp: int, rc: int) -> dict:
    return dict(zip(keys, divmod(rp, rc)))
Eaos = Euclids_algorithm_one_step

while divmod_list[-1]['rem'] != 1:
    rp, rc = divmod_list[-2]['rem'], divmod_list[-1]['rem'] 
    divmod_list.append(Eaos(rp, rc))
else:
    print(divmod_list[2:])
    
#print(private_key)

[{'quo': 18389, 'rem': 17987}, {'quo': 3, 'rem': 11576}, {'quo': 1, 'rem': 6411}, {'quo': 1, 'rem': 5165}, {'quo': 1, 'rem': 1246}, {'quo': 4, 'rem': 181}, {'quo': 6, 'rem': 160}, {'quo': 1, 'rem': 21}, {'quo': 7, 'rem': 13}, {'quo': 1, 'rem': 8}, {'quo': 1, 'rem': 5}, {'quo': 1, 'rem': 3}, {'quo': 1, 'rem': 2}, {'quo': 1, 'rem': 1}]


In [26]:
### Creating Bézout's identity

x_p, x_e = sp.var(['x_p', 'x_e'])

eqns = [x_p, x_e]
for i in range(2,len(divmod_list)):
    eqns.append(eqns[-2] - divmod_list[i]['quo']*eqns[-1])

private_key = int(eqns[-1].coeff(x_e))
print(f"{private_key=}")

private_key=459437633


### Decryption

In [27]:
#decrypted_pw = tuple(enum**private_key % public_key[0] for enum in encrypted_pw)
# 上の最も素朴なべき乗 enum**private_key は時間がかかりすぎる。よって、以下の組み込み関数 pow() (2進展開し、平方と乗法を繰り返す)で高速化。

decrypted_pw = tuple(pow(enum, private_key, public_key[0]) for enum in encrypted_pw)
print(decrypted_pw)
if decrypted_pw == pw:
    print(f"Decryption Successful")

(2120, 6714, 2126, 3784)
Decryption Successful


In [28]:
### 中国剰余定理の利用バージョン

qinv = pow(q, -1, p)

decrypted_pw = []
for enum in encrypted_pw:
    enum_p, enum_q = tuple(pow(enum % prm, (private_key % (prm-1)), prm) for prm in [p,q])
    h = qinv*(enum_p-enum_q) % p
    decrypted_pw.append(enum_q + h*q)

print(decrypted_pw)

[2120, 6714, 2126, 3784]


# Scratch

In [30]:
import time
import statistics
import secrets

# ====== RSA core ======
def rsa_decrypt_naive(c, d, n):
    # 素朴版：m = c^d mod n
    return pow(c, d, n)

def rsa_decrypt_crt(c, p, q, d):
    # CRT 版（Python 3.8+）
    # 逆元 qinv = q^{-1} mod p
    qinv = pow(q, -1, p)
    dp = d % (p - 1)
    dq = d % (q - 1)

    m_p = pow(c % p, dp, p)
    m_q = pow(c % q, dq, q)
    h = (qinv * (m_p - m_q)) % p
    return m_q + h * q

def rsa_decrypt_crt_with_params(c, p, q, dmp1, dmq1, iqmp):
    # OpenSSL 風の事前計算済みパラメータ（推奨）
    m_p = pow(c % p, dmp1, p)
    m_q = pow(c % q, dmq1, q)
    h = (iqmp * (m_p - m_q)) % p
    return m_q + h * q

# ====== Key generation (toy; for demo) ======
# 実用鍵の生成は外部ライブラリ（例：cryptography）推奨。
# ここではデモ用に安全ではない簡略生成を行います。
def generate_toy_rsa_key(bits=1024):
    # 乱数から素数を作る簡易版（Miller-Rabin）
    # 本番用途不可。速度比較のデモ専用。
    def is_probable_prime(n, k=32):
        if n < 2:
            return False
        # 小さな素数で試し割り
        small_primes = [2,3,5,7,11,13,17,19,23,29]
        for p in small_primes:
            if n % p == 0:
                return n == p
        # n-1 = d * 2^r
        r, d = 0, n - 1
        while d % 2 == 0:
            r += 1
            d //= 2
        for _ in range(k):
            a = secrets.randbelow(n-3) + 2  # in [2, n-2]
            x = pow(a, d, n)
            if x == 1 or x == n - 1:
                continue
            for __ in range(r - 1):
                x = pow(x, 2, n)
                if x == n - 1:
                    break
            else:
                return False
        return True

    def gen_prime(bits):
        while True:
            # 最上位/最下位ビットを立てる
            candidate = secrets.randbits(bits) | (1 << (bits - 1)) | 1
            if is_probable_prime(candidate):
                return candidate

    half = bits // 2
    p = gen_prime(half)
    q = gen_prime(half)
    while p == q:
        q = gen_prime(half)

    n = p * q
    phi = (p - 1) * (q - 1)

    # 一般的な公開指数 e
    e = 65537
    # e と phi は互いに素であるべき
    # まれに非互いに素の場合があるので修正
    def gcd(a, b):
        while b:
            a, b = b, a % b
        return a
    if gcd(e, phi) != 1:
        # フォールバック（あまり起きない）
        e = 3
        while gcd(e, phi) != 1:
            e += 2

    # 秘密指数 d
    d = pow(e, -1, phi)  # Python 3.8+

    # 事前計算パラメータ
    dmp1 = d % (p - 1)
    dmq1 = d % (q - 1)
    iqmp = pow(q, -1, p)

    return {
        "p": p, "q": q, "n": n, "e": e, "d": d,
        "dmp1": dmp1, "dmq1": dmq1, "iqmp": iqmp
    }

# ====== Benchmark ======
def bench_once(func, *args):
    t0 = time.perf_counter()
    _ = func(*args)
    t1 = time.perf_counter()
    return t1 - t0

def benchmark_rsa_decrypt(iterations=20, bits=2048):
    key = generate_toy_rsa_key(bits=bits)
    n, e, d = key["n"], key["e"], key["d"]
    p, q = key["p"], key["q"]
    dmp1, dmq1, iqmp = key["dmp1"], key["dmq1"], key["iqmp"]

    # ランダム平文→暗号文を作成
    m_plain = secrets.randbelow(n - 2) + 2  # avoid trivial 0/1
    c = pow(m_plain, e, n)

    # ウォームアップ
    for _ in range(3):
        rsa_decrypt_naive(c, d, n)
        rsa_decrypt_crt(c, p, q, d)
        rsa_decrypt_crt_with_params(c, p, q, dmp1, dmq1, iqmp)

    # 計測
    times_naive = [bench_once(rsa_decrypt_naive, c, d, n) for _ in range(iterations)]
    times_crt   = [bench_once(rsa_decrypt_crt, c, p, q, d) for _ in range(iterations)]
    times_crtp  = [bench_once(rsa_decrypt_crt_with_params, c, p, q, dmp1, dmq1, iqmp) for _ in range(iterations)]

    summary = {
        "naive_median_ms": statistics.median(times_naive) * 1000,
        "crt_median_ms": statistics.median(times_crt) * 1000,
        "crt_params_median_ms": statistics.median(times_crtp) * 1000,
        "speedup_crt_vs_naive": (statistics.median(times_naive) / statistics.median(times_crt)),
        "speedup_crtp_vs_naive": (statistics.median(times_naive) / statistics.median(times_crtp)),
        "bits": bits
    }
    return summary

if __name__ == "__main__":
    # 反復回数と鍵サイズは環境に応じて調整してください
    RSA_bits = 1024*4
    result = benchmark_rsa_decrypt(iterations=30, bits=RSA_bits)
    print(f"RSA bits       : {result['bits']}")
    print(f"Naive median   : {result['naive_median_ms']:.3f} ms")
    print(f"CRT median     : {result['crt_median_ms']:.3f} ms  (speedup x{result['speedup_crt_vs_naive']:.2f})")
    print(f"CRT+params med.: {result['crt_params_median_ms']:.3f} ms  (speedup x{result['speedup_crtp_vs_naive']:.2f})")

RSA bits       : 4096
Naive median   : 102.776 ms
CRT median     : 28.835 ms  (speedup x3.56)
CRT+params med.: 28.606 ms  (speedup x3.59)


In [53]:
order = 2**4 #2**5 #2**10 #
p,q = 5, 7
print(p, q)

n = p*q
phi = (p-1)*(q-1)
e = 13

public_key = (n,e)
print(f"{public_key=}")

### Euclid's algorithm

keys = ['quo', 'rem']
divmod_list = [dict(zip(keys, (0, val))) for val in [phi, e]]

def Euclids_algorithm_one_step(rp: int, rc: int) -> dict:
    return dict(zip(keys, divmod(rp, rc)))
Eaos = Euclids_algorithm_one_step

while divmod_list[-1]['rem'] != 1:
    rp, rc = divmod_list[-2]['rem'], divmod_list[-1]['rem'] 
    divmod_list.append(Eaos(rp, rc))
else:
    pass #print(divmod_list[2:])
    
#print(private_key)

### Creating Bézout's identity

x_p, x_e = sp.var(['x_p', 'x_e'])

eqns = [x_p, x_e]
for i in range(2,len(divmod_list)):
    eqns.append(eqns[-2] - divmod_list[i]['quo']*eqns[-1])

private_key = int(eqns[-1].coeff(x_e))
print(f"{e=},{private_key=}")

5 7
public_key=(35, 13)
e=13,private_key=-11


In [54]:
phi, phi%(q-1)

(24, 0)

In [55]:
pw = 7
display(pw)

7

In [56]:
encrypted_pw = pw**public_key[1] % public_key[0]
print(f"{encrypted_pw=}")

encrypted_pw=7


### Decryption

In [57]:
#decrypted_pw = tuple(enum**private_key % public_key[0] for enum in encrypted_pw)
# 上の最も素朴なべき乗 enum**private_key は時間がかかりすぎる。よって、以下の組み込み関数 pow() (2進展開し、平方と乗法を繰り返す)で高速化。

decrypted_pw = tuple(pow(enum, private_key, public_key[0]) for enum in encrypted_pw)
print(decrypted_pw)
if decrypted_pw == pw:
    print(f"Decryption Successful")

TypeError: 'int' object is not iterable